<img src="../images/ryzhkov_qvp_reproduction.png" width=700 alt="QVP plot"></img>

# QVP Analysis: Traditional vs ARCO Workflows

This notebook demonstrates the benefits of **Analysis-Ready Cloud-Optimized (ARCO)** radar data by reproducing **Figure 4** from [Ryzhkov et al. (2016)](https://doi.org/10.1175/JTECH-D-15-0020.1) using two contrasting approaches.

As defined by [Abernathey et al. (2021)](https://doi.org/10.1109/MCSE.2021.3059437), ARCO data represents the "gold standard" for cloud-native scientific data repositories, combining:
- **Analysis-Ready**: Data prepared for immediate exploration without preprocessing
- **Cloud-Optimized**: Chunked formats (Zarr) enabling efficient HTTP-based access

```{list-table} Workflow Comparison Overview
:header-rows: 1
:widths: 25 35 35

* - Aspect
  - Traditional Workflow
  - ARCO Data Streaming
* - **Data Access**
  - Download full files
  - Stream chunks on-demand
* - **I/O Pattern**
  - Download → Decode → Process
  - Lazy open → Select → Stream
* - **Sweeps Loaded**
  - All 17 sweeps per file
  - Only requested sweep(s)
* - **Variables Loaded**
  - All variables in file
  - Only requested variables
* - **Time Selection**
  - Parse filenames, filter list
  - Native `vcp_time.sel()` slice
* - **Throughput**
  - Limited by download + decode
  - Scales with parallelism
```

**Scientific Goal**: Compute Quasi-Vertical Profiles (QVPs) of Z, Z_DR, ρ_HV, and Φ_DP for the May 20, 2011 MCS observed by KVNX radar during MC3E.

---

```{admonition} Prerequisites
:class: note

This notebook assumes familiarity with the basics covered in **[Notebook 1 — Open NEXRAD radar archives in 5 lines with radar-datatree](1.NEXRAD-KLOT-Demo)**:
- Connecting to cloud storage with Icechunk
- Opening a DataTree with `engine="rustytree"`
- Time-based selection with `.sel()`

If you're new to radar-datatree, start there first.
```

## What is a Quasi-Vertical Profile (QVP)?

A **Quasi-Vertical Profile (QVP)** is a technique for creating time-height displays of polarimetric radar variables by **azimuthally averaging** data at high elevation angles.

### The Concept

At high elevation angles (e.g., 19.5°), the radar beam rises quickly with range:

```
         ↗ Beam at 19.5° elevation
        /
       /  Height increases with range
      /
    [RADAR]
```

By averaging all azimuth angles (360° rotation) at each range gate, we collapse the horizontal variability and create a **vertical profile** of the atmosphere above the radar:

$$\text{QVP}(r, t) = \frac{1}{N_{\theta}} \sum_{\theta=0}^{360°} Z(r, \theta, t)$$

where $r$ is range (converted to height), $\theta$ is azimuth, and $t$ is time.

### Why QVPs?

QVPs are powerful for:
- **Monitoring precipitation structure**: See the melting layer, dendritic growth zones, and aggregation signatures
- **Hydrometeor classification**: Different particle types have distinct polarimetric signatures
- **Storm evolution**: Time-height plots reveal temporal changes in precipitation processes

The figure at the top of this notebook (from Ryzhkov et al. 2016) shows QVPs of Z, Z_DR, ρ_HV, and Φ_DP—we'll reproduce it using both traditional and ARCO workflows.

## Study Parameters

We'll analyze an expanded time window to capture more storm evolution:
- **Radar**: KVNX (Vance Air Force Base, Oklahoma)
- **Date**: May 20, 2011 (MC3E field campaign)
- **Time Window**: 08:30 - 12:30 UTC (4 hours)
- **Elevation**: 19.5° (highest WSR-88D elevation, sweep_16)
- **Variables**: DBZH, ZDR, RHOHV, PHIDP

In [ ]:
import sys
import time
import tracemalloc  # For memory tracking
import warnings
from pathlib import Path

# Suppress FutureWarning from xradar/xarray
warnings.filterwarnings("ignore", category=FutureWarning)

# Ensure demo_functions is importable when sphinx-build runs from docs/.
sys.path.insert(0, str(Path("../notebooks").resolve()))

import cmweather  # noqa: F401
import icechunk as ic
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from demo_functions import (
    assert_qvp_equivalence,
    calculate_chunk_metrics,
    compute_qvp,
    get_repo_config,
    list_nexrad_files_with_sizes,
    nexrad_download_with_size,
    plot_workflow_comparison,
    print_arco_summary,
    print_chunk_analysis,
    print_traditional_summary,
    print_workflow_comparison,
    ryzhkov_figure,
)

# Study parameters - expanded time window (4 hours)
RADAR = "KVNX"
START_TIME = "2011-05-20 08:30"
END_TIME = "2011-05-20 12:30"
VARIABLES = ["DBZH", "ZDR", "RHOHV", "PHIDP"]
# 19.5° elevation, the angle Ryzhkov et al. (2016) use for QVPs. Both
# the file-based loop below and the ARCO path target this exact sweep
# so they're guaranteed to operate on identical underlying scans.
SELECTED_SWEEP = "sweep_16"

# Metrics collection
metrics = {"traditional": {}, "arco": {}}

---
## Approach 1: Traditional File-Based Workflow

The conventional approach requires:
1. **Discovery**: Query S3 bucket to list files in time range
2. **Download**: Fetch each compressed Level II file
3. **Decode**: Parse binary format into datatree
4. **Extract**: Select the 19.5° sweep from each volume
5. **Aggregate**: Manually concatenate along time
6. **Compute**: Calculate QVP for each timestep

### Step 1: Discover files in the time range

In [ ]:
# List all NEXRAD files with actual sizes from AWS S3 using fsspec
t0 = time.time()
nexrad_files = list_nexrad_files_with_sizes(
    radar=RADAR, start_time=START_TIME, end_time=END_TIME
)
metrics["traditional"]["discovery_time"] = time.time() - t0
metrics["traditional"]["n_files"] = len(nexrad_files)
metrics["traditional"]["total_size_bytes"] = sum(f["size"] for f in nexrad_files)
metrics["traditional"]["total_size_mb"] = metrics["traditional"]["total_size_bytes"] / (
    1024 * 1024
)

print(f"Found {len(nexrad_files)} files in the time range")
print(
    f"First file: {nexrad_files[0]['path'].split('/')[-1]} ({nexrad_files[0]['size'] / 1e6:.1f} MB)"
)
print(
    f"Last file:  {nexrad_files[-1]['path'].split('/')[-1]} ({nexrad_files[-1]['size'] / 1e6:.1f} MB)"
)
print(f"\nTotal data volume: {metrics['traditional']['total_size_mb']:.1f} MB")
print(f"Discovery time: {metrics['traditional']['discovery_time']:.2f}s")

### Step 2: Download, decode, and extract sweeps

This is the most expensive part - each file must be:
- Downloaded from S3 (~30 MB compressed per file)
- Decompressed
- Decoded from binary NEXRAD format
- Parsed into xarray structures

**Note**: We'll process a subset to demonstrate the workflow without excessive wait times.

In [ ]:
# Process ALL files - download, decode, extract SELECTED_SWEEP, store in RAM.
# This demonstrates the full traditional workflow with actual memory usage.

t0 = time.time()
tracemalloc.start()  # Start memory tracking

sweep_datasets = []  # Store actual xarray datasets in memory
download_times = []
bytes_downloaded = 0

print(f"Processing all {len(nexrad_files)} files...")
print("-" * 60)

for i, file_info in enumerate(nexrad_files):
    try:
        # Download and decode
        t_download = time.time()
        dtree_single, size_bytes = nexrad_download_with_size(file_info["path"])
        download_times.append(time.time() - t_download)
        bytes_downloaded += size_bytes

        # Extract and LOAD the chosen sweep into memory. KeyError if a file
        # is in a VCP that lacks SELECTED_SWEEP — we want loud failure here
        # rather than silently averaging across mixed elevations.
        ds = dtree_single[SELECTED_SWEEP].ds.load()  # .load() forces into RAM

        # Add time coordinate for concatenation
        sweep_time = ds.time.values[0]
        ds = ds.expand_dims({"vcp_time": [sweep_time]})

        sweep_datasets.append(ds)

        if (i + 1) % 10 == 0 or (i + 1) == len(nexrad_files):
            current_mem, peak_mem = tracemalloc.get_traced_memory()
            print(
                f"  [{i + 1}/{len(nexrad_files)}] {file_info['path'].split('/')[-1]} - "
                f"RAM: {current_mem / 1e6:.0f} MB (peak: {peak_mem / 1e6:.0f} MB)"
            )

    except Exception as e:
        print(f"  [{i + 1}/{len(nexrad_files)}] Failed: {e}")

# Get final memory stats
current_mem, peak_mem = tracemalloc.get_traced_memory()
tracemalloc.stop()

metrics["traditional"]["total_time"] = time.time() - t0
metrics["traditional"]["avg_download_time"] = np.mean(download_times)
metrics["traditional"]["files_processed"] = len(sweep_datasets)
metrics["traditional"]["bytes_downloaded"] = bytes_downloaded
metrics["traditional"]["peak_memory_mb"] = peak_mem / 1e6
metrics["traditional"]["final_memory_mb"] = current_mem / 1e6

print("-" * 60)
print(
    f"Downloaded and loaded {len(sweep_datasets)} files in {metrics['traditional']['total_time']:.1f}s"
)
print(f"Network transfer: {bytes_downloaded / 1e6:.1f} MB (compressed)")
print(f"Peak RAM usage: {peak_mem / 1e6:.0f} MB")
print(
    f"Average time per file: {metrics['traditional']['total_time'] / len(sweep_datasets):.2f}s"
)

### Traditional Workflow: Estimated Full Processing Time

In [ ]:
# Concatenate all sweep datasets and compute QVPs (traditional approach)
print("Concatenating datasets along time dimension...")
t0_concat = time.time()

tracemalloc.start()
ds_traditional = xr.concat(sweep_datasets, dim="vcp_time")
concat_time = time.time() - t0_concat

# Compute QVPs from the concatenated dataset using the same `compute_qvp`
# helper as the ARCO path. Identical transforms (dB-aware azimuth mean +
# range→height rebinning) on both paths means the resulting QVPs can be
# compared directly value-for-value.
t0_qvp = time.time()
qvp_traditional = {
    var: compute_qvp(ds_traditional, var=var)
    for var in VARIABLES
    if var in ds_traditional.data_vars
}
qvp_compute_time = time.time() - t0_qvp

_, peak_mem = tracemalloc.get_traced_memory()
tracemalloc.stop()

metrics["traditional"]["concat_time"] = concat_time
metrics["traditional"]["qvp_compute_time"] = qvp_compute_time
metrics["traditional"]["concat_peak_memory_mb"] = peak_mem / 1e6
metrics["traditional"]["total_workflow_time"] = (
    metrics["traditional"]["total_time"] + concat_time + qvp_compute_time
)

print_traditional_summary(metrics, ds_traditional)

---
## Approach 2: Radar DataTree (ARCO) with Data Streaming

The cloud-native approach with Radar DataTree uses **on-demand data streaming**:
1. **Connect**: Open Icechunk repository (metadata only, no data transfer)
2. **Navigate**: Browse VCPs and sweeps via hierarchical structure  
3. **Select**: Slice by native `vcp_time` dimension (lazy, no data yet)
4. **Stream**: Data chunks are fetched on-demand during computation

**Key difference**: No file downloads, no decoding overhead. Data streams directly from cloud storage as needed.

### Step 1: Connect to Icechunk repository

In [ ]:
%%time
t0 = time.time()

# Configure S3-compatible storage (OSN)
storage = ic.s3_storage(
    bucket="nexrad-arco",
    prefix="KVNX",
    endpoint_url="https://umn1.osn.mghpcc.org",
    anonymous=True,
    force_path_style=True,
    region="us-east-1",
)

# Connect to repository with optimized config
repo_config = get_repo_config()
repo = ic.Repository.open(storage, config=repo_config)
session = repo.readonly_session("main")

metrics["arco"]["connect_time"] = time.time() - t0
print(f"Connected to Icechunk repository in {metrics['arco']['connect_time']:.2f}s")

Open the radar datatree using xarray.

In [ ]:
%%time
t0 = time.time()

# Open only the sweep we need across all VCPs. The `/*/sweep_16` glob
# trims the tree from "every VCP × every sweep" down to just SELECTED_SWEEP
# (the 19.5° elevation Ryzhkov et al. 2016 use for QVPs). See
# Notebook 1's "Sweep_0 from every VCP" section for the same pattern.
dtree = xr.open_datatree(
    session.store,
    engine="rustytree",
    group=f"/*/{SELECTED_SWEEP}",
    chunks={},
)

metrics["arco"]["open_datatree_time"] = time.time() - t0
print(f"Opened DataTree in {metrics['arco']['open_datatree_time']:.2f}s")

### Step 2: Explore the trimmed hierarchy

Because we passed `group="/*/sweep_16"`, the DataTree contains only the sweep we asked for, organized under each VCP that has it. Listing the children shows the VCPs the radar cycled through during the storm.

In [ ]:
list(dtree.children)

Each VCP's `sweep_16` is the 19.5° elevation cut. The QVP-target VCP for the May 2011 KVNX MCS is `VCP-12` (precipitation mode); pull its sweep directly:

In [ ]:
ds_qvp = dtree[f"/VCP-12/{SELECTED_SWEEP}"].ds
ds_qvp

### Step 3: Select time window and compute QVPs

Unlike the traditional workflow that requires iterating through files, we can directly slice the data using the native `vcp_time` dimension. The entire 4-hour window is accessed with a single selection.

In [ ]:
# Select the 4-hour time window
# With ARCO format, this is a simple slice operation - no file iteration needed
ds_qvp_selected = ds_qvp.sel(
    vcp_time=slice(START_TIME.replace(" ", "T"), END_TIME.replace(" ", "T"))
)
print(
    f"Time range: {ds_qvp_selected.vcp_time.min().values} to {ds_qvp_selected.vcp_time.max().values}"
)
print(f"Number of timesteps: {len(ds_qvp_selected.vcp_time)}")

In [ ]:
# Calculate ARCO streaming metrics — data fetched on-demand
n_chunks, arco_bytes, chunk_details = calculate_chunk_metrics(
    ds_qvp_selected, VARIABLES
)

# 17 sweeps per VCP-12 NEXRAD Level II volume; the QVP uses the highest one
# (sweep_16 ≈ 19.5°), so traditional downloads everything and discards 16/17.
N_TOTAL_SWEEPS = 17

# Side effect: stashes uncompressed_mb on both branches + chunk_details under
# metrics so print_workflow_comparison reads from a single source of truth.
print_chunk_analysis(
    metrics,
    chunk_details,
    n_chunks,
    arco_bytes,
    selected_sweep=SELECTED_SWEEP,
    n_total_sweeps=N_TOTAL_SWEEPS,
)

metrics["arco"]["n_chunks"] = n_chunks

In [ ]:
%%time
t0 = time.time()

# Compute QVP for multiple radar variables (vectorized across all timesteps)
qvp_data = {}

for var in VARIABLES:
    if var in ds_qvp_selected.data_vars:
        qvp_data[var] = compute_qvp(ds_qvp_selected, var=var).compute()
        print(f"Computed QVP for {var}")
    else:
        print(f"Warning: Variable {var} not found in dataset")

metrics["arco"]["qvp_compute_time"] = time.time() - t0
metrics["arco"]["timesteps"] = len(ds_qvp_selected.vcp_time)
print(
    f"\nProcessed {metrics['arco']['timesteps']} timesteps in {metrics['arco']['qvp_compute_time']:.1f}s"
)

### Sanity check: traditional and ARCO QVPs agree

Both paths read the same sweep_16 bytes and pass them through the same `compute_qvp` helper, so the resulting QVPs should match within floating-point noise. This block enforces the equivalence directly — if either workflow drifted (e.g., a misaligned time slice, a chunk-decoding bug, a unit handling regression), the assertion fires.

In [ ]:
TOLERANCES = {
    "DBZH": (0.1, "dB"),
    "ZDR": (0.1, "dB"),
    "RHOHV": (0.05, ""),
    "PHIDP": (0.5, "deg"),
}

assert_qvp_equivalence(qvp_traditional, qvp_data, VARIABLES, TOLERANCES)

In [ ]:
# Reproduce Ryzhkov et al. (2016) Figure 4: QVP time-height cross-sections
ryzhkov_figure(qvp_data["DBZH"], qvp_data["ZDR"], qvp_data["RHOHV"], qvp_data["PHIDP"])
plt.savefig("ryzhkov_qvp_reproduction.png", dpi=150, bbox_inches="tight")
print("Figure saved as 'ryzhkov_qvp_reproduction.png'")

### ARCO Workflow: Total Time

In [ ]:
metrics["arco"]["total_time"] = (
    metrics["arco"]["connect_time"]
    + metrics["arco"]["open_datatree_time"]
    + metrics["arco"]["qvp_compute_time"]
)

print_arco_summary(metrics)

---
## Performance Comparison: Traditional vs ARCO

Now let's compare the two approaches quantitatively.

In [ ]:
metrics["speedup"] = (
    metrics["traditional"]["total_workflow_time"] / metrics["arco"]["total_time"]
)
metrics["data_reduction"] = (
    metrics["traditional"]["uncompressed_mb"] / metrics["arco"]["uncompressed_mb"]
)
metrics["traditional"]["throughput_mbs"] = (
    metrics["arco"]["uncompressed_mb"] / metrics["traditional"]["total_workflow_time"]
)
metrics["arco"]["throughput_mbs"] = (
    metrics["arco"]["uncompressed_mb"] / metrics["arco"]["total_time"]
)
metrics["throughput_gain"] = (
    metrics["arco"]["throughput_mbs"] / metrics["traditional"]["throughput_mbs"]
)

# Surfaced for the plot cell below.
speedup = metrics["speedup"]

print_workflow_comparison(
    metrics,
    selected_sweep=SELECTED_SWEEP,
    n_total_sweeps=N_TOTAL_SWEEPS,
)

In [ ]:
plot_workflow_comparison(
    metrics,
    save_path="workflow_comparison_cleaned.png",
)
plt.show()

---
## Key Takeaways

This analysis demonstrates the principles outlined in [Abernathey et al. (2021)](https://doi.org/10.1109/MCSE.2021.3059437) for cloud-native scientific data repositories.

### ARCO Principles Applied to Radar Data

**Analysis-Ready, Cloud-Optimized (ARCO)** data, as defined by Abernathey et al., represents the "gold standard" for cloud-native repositories. Our Radar DataTree implementation embodies these principles:

1. **Elimination of Scientific Toil**: The traditional workflow requires "manual, repetitive, automatable" work—downloading files, parsing filenames, looping through data, concatenating arrays. ARCO eliminates this toil:
   ```python
   # Traditional: ~50 lines of file iteration, download, decode, concat
   # ARCO: Single declarative selection
   ds.sel(vcp_time=slice("2011-05-20T08:30", "2011-05-20T12:30")).compute()
   ```

2. **Cloud-Optimized Format (Zarr/Icechunk)**: Data organized in chunks that enable:
   - HTTP-based access without filesystem assumptions
   - Efficient subsetting (only requested chunks are fetched)
   - Parallel distributed processing

3. **FAIR Principles Enhanced**:
   - **Findable**: Hierarchical DataTree structure (VCP → sweep → variable)
   - **Accessible**: Global namespace (`s3://`) enables portable, reproducible code
   - **Interoperable**: CF-compliant metadata, standard xarray interface
   - **Reusable**: Versioned with Icechunk for reproducibility

4. **Selective Data Access**: The key advantage of ARCO is **not compression** but **selective access**. Traditional workflows download entire files (all sweeps, all variables) even when only a subset is needed. ARCO streams exactly the chunks required—nothing more.

### Performance Summary

We measure **effective throughput** = useful data delivered / total workflow time.

Both workflows deliver the same 146 MB of useful data (4 variables × 55 timesteps × 1 sweep) for QVP analysis:

```{list-table} Measured Performance (Fair Comparison)
:header-rows: 1
:widths: 35 25 25 15

* - Metric
  - Traditional
  - ARCO Streaming
  - Gain
* - Total Time
  - ~260 seconds
  - ~8 seconds
  - **32x faster**
* - Peak RAM
  - 3,274 MB
  - 146 MB
  - **22x less**
* - Network Transfer
  - 809 MB (gzip)
  - 146 MB (uncompressed)
  - **5.5x less**
* - Effective Throughput
  - 0.56 MB/s
  - 18 MB/s
  - **32x higher**
```

**Key insight**: ARCO's advantage comes from **selective access**—streaming only the 146 MB of chunks needed instead of downloading 809 MB of compressed files (which decompress to ~3,200 MB in RAM). Even without compression in the ARCO store, we transfer 5.5x less data because we only fetch what we need.

### Scalability

The same ARCO pattern scales from hours to years:
- **4 hours** (this demo): ~8 seconds
- **1 day**: ~48 seconds (estimated)
- **1 month**: ~24 minutes (estimated)
- **1 year**: ~5 hours (estimated)

Traditional file-based workflows scale linearly with data volume. ARCO maintains consistent per-chunk access patterns regardless of total dataset size.

### When to Use Each Approach

```{list-table} Recommended Workflow by Use Case
:header-rows: 1
:widths: 50 50

* - Use Case
  - Recommended Approach
* - Single file exploration (local)
  - Traditional
* - Time-series analysis (single sweep)
  - **ARCO streaming**
* - Multi-sweep analysis
  - **ARCO streaming** (select sweeps needed)
* - Large-scale processing (months/years)
  - **ARCO streaming**
* - Reproducible research
  - **ARCO** (versioned with Icechunk)
* - Machine learning training
  - **ARCO** (high-throughput chunk access)
```

## References

* Abernathey, R.P., T. Augspurger, A. Banihirwe, C.C. Blackmon-Luca, T.J. Crone, C.L. Gentemann, J.J. Hamman, N. Henderson, C. Lepore, T.A. McCaie, N.H. Robinson, and R.P. Signell, 2021: Cloud-Native Repositories for Big Scientific Data. *Computing in Science & Engineering*, **23**, 26–35, https://doi.org/10.1109/MCSE.2021.3059437.

* Ryzhkov, A., P. Zhang, H. Reeves, M. Kumjian, T. Tschallener, S. Trömel, and C. Simmer, 2016: Quasi-Vertical Profiles—A New Way to Look at Polarimetric Radar Data. *J. Atmos. Oceanic Technol.*, **33**, 551–562, https://doi.org/10.1175/JTECH-D-15-0020.1.

* Wilkinson, M.D., et al., 2016: The FAIR Guiding Principles for scientific data management and stewardship. *Scientific Data*, **3**, 160018, https://doi.org/10.1038/sdata.2016.18.